# Aula 11 — Streaming: ver o grafo executando

**Objetivo:** trocar o `invoke` (só o final) pelo `stream` (a execução ao vivo)
nos 3 modos — e entender qual modo alimenta cada necessidade: log, foto e tokens.

**Pré-requisitos:** Aula 05 (o loop ReAct) e Aula 06 (`checkpointer` + `thread`).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` os tokens são reais; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: rastreio de entrega

Você comprou um pastel pelo delivery. Três jeitos de acompanhar:

| Delivery | No LangGraph | O que chega |
|---|---|---|
| 'saiu para entrega', 'chegou' | **`stream_mode='updates'`** | o DELTA de cada nó (`{'tools': {...}}`) |
| foto atual do pedido | **`stream_mode='values'`** | o estado COMPLETO após cada nó |
| campainha a cada novidade | **`stream_mode='messages'`** | tuplas (pedacinho, metadata com `langgraph_node`) |

E a diferença raiz: `invoke` é a **foto final**; `stream` é um **gerador
preguiçoso** — nada executa até você iterar, e depois de consumido ele esvazia.

In [ ]:
# CÉLULA 2 — o agente ReAct com memória (base das demos)
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

SISTEMA = 'Você é um assistente que faz contas. Responda em português, de forma curta.'
MODELO = 'openai/gpt-oss-20b'

def add(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

TOOLS = [add]

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline (2 falas gravadas).')
    return FakeMessagesListChatModel(responses=[
        AIMessage(content='', tool_calls=[
            {'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'c1', 'type': 'tool_call'}]),
        AIMessage(content='O resultado é 7.', tool_calls=[]),
    ])

# O LLM nasce UMA vez aqui fora (criar dentro do nó, com Fake, zera a
# sequência a cada volta e o loop nunca termina — Aula 06 ensinou).
LLM = get_llm()

def assistant(state: MessagesState) -> dict:
    return {'messages': [LLM.invoke(
        [SystemMessage(content=SISTEMA)] + state['messages'])]}

b = StateGraph(MessagesState)
b.add_node('assistant', assistant)
b.add_node('tools', ToolNode(TOOLS))
b.add_edge(START, 'assistant')
b.add_conditional_edges('assistant', tools_condition)
b.add_edge('tools', 'assistant')
reagente = b.compile(checkpointer=MemorySaver())
print('agente com memória montado.')

## 2. Conceito: `invoke` entrega, `stream` transmite

- `invoke(input)` bloqueia até o `END` e devolve **um** estado final.
- `stream(input, config, stream_mode=...)` devolve **um gerador**: cada `for`
  entrega um pedaço da execução. Nada roda antes do 1º `next` (preguiçoso) e,
  depois de consumido, ele **esvazia** (segundo `for` vem vazio).
- `config` com `thread_id` continua obrigatório com checkpointer — igual ao
  `invoke` (Aula 06). Cada modo abaixo usa thread própria para não misturar.
- `astream`/`astream_events` são as versões `async` (mesma lógica, com `await`;
  aqui usamos `await` direto na célula (padrão Jupyter).

In [ ]:
# CÉLULA 3 — updates: um delta por nó (rode e conte)
for chunk in reagente.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                             {'configurable': {'thread_id': 'u1'}},
                             stream_mode='updates'):
    print(list(chunk.keys()), '->', str(chunk)[:100].replace(chr(10), ' '))
print()
print('Leitura: 3 deltas = 3 passagens por nós (assistant, tools, assistant).')

## 3. Dissecando os 3 modos (o que cada chunk carrega)

- **`updates`**: `{'assistant': {'messages': [...]}}` — SÓ o que o nó produziu.
  Ideal para log/barra de progresso ('passo 2 de 3: tools').
- **`values`**: o estado **inteiro** após cada nó (`1, 2, 3, 4 msgs`). Ideal
  para 'foto atual' (UI que mostra a conversa crescendo).
- **`messages`**: tuplas `(pedacinho, metadata)`; `metadata['langgraph_node']`
  diz quem emitiu. Ideal para 'digitando...' por nó e para filtrar tokens.

In [ ]:
# CÉLULA 4 — values: o estado crescendo (rode e confira 1..4)
for chunk in reagente.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                             {'configurable': {'thread_id': 'u2'}},
                             stream_mode='values'):
    print('estado com', len(chunk['messages']), 'msgs')
print()
print('Leitura: foto após cada nó. O último é igual ao invoke.')

In [ ]:
# CÉLULA 5 — messages: tupla + origem (rode e confira)
for pedaco, meta in reagente.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                                     {'configurable': {'thread_id': 'u3'}},
                                     stream_mode='messages'):
    print('[' + type(pedaco).__name__ + ' do nó ' + str(meta.get('langgraph_node')) + ']',
          repr(pedaco.content)[:60])
print()
print('Leitura: com Groq real, os contents vêm pingando (tokens);')
print('no Fake, vêm inteiros de uma vez — a estrutura (tupla + nó) é igual.')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Gerador esgota: segunda passada vem vazia

`stream` não é lista — é gerador de uso único. Precisa dos chunks 2x?
Converta com `list()` **uma vez** e reuse a lista.

In [ ]:
# CÉLULA 6 — gerador de uso único (100% offline na lógica)
gen = reagente.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                      {'configurable': {'thread_id': 'u4'}},
                      stream_mode='updates')
print('tipo:', type(gen).__name__)
primeira = list(gen)
print('1ª passada:', len(primeira), 'chunks | 2ª passada:', len(list(gen)))
assert len(primeira) == 3 and list(gen) == []
print('OK — consumiu, esvaziou. Guarde em lista se precisar reler.')

### 4b. `astream`: o mesmo `stream`, com `await`

Em servidor web (FastAPI) e UI reativa, o loop de eventos não pode bloquear:
usa-se `astream`. Aqui rodamos com `asyncio.run()` para funcionar em qualquer
lugar (no Jupyter puro, `await` direto também vale).

In [ ]:
# CÉLULA 7 — astream (async)

async def coleta():
    return [c async for c in reagente.astream(
        {'messages': [HumanMessage(content='Some 3 e 4')]}, 
        {'configurable': {'thread_id': 'u5'}},
        stream_mode='updates')]

chunks = await coleta()
print([list(c.keys()) for c in chunks])
assert [list(c.keys()) for c in chunks] == [['assistant'], ['tools'], ['assistant']]
print('OK — idêntico ao stream síncrono.')

### 4c. `astream_events`: filtrar tokens por nó

`astream_events(..., version='v2')` emite TUDO (nó, LLM, tool) como eventos
tipados. O filtro `on_chat_model_stream` + `metadata['langgraph_node']` seleciona
só os tokens do LLM do nó que interessa — é assim que um chat mostra
'digitando...' só do assistente, ignorando o resto.

In [ ]:
# CÉLULA 8 — tokens do assistant (estrutura offline, texto real com Groq)

async def tokens_do_assistant():
    toks = []
    async for ev in reagente.astream_events(
            {'messages': [HumanMessage(content='Some 3 e 4. Responda com frase.')]},
            {'configurable': {'thread_id': 'u6'}}, version='v2'):
        if ev['event'] == 'on_chat_model_stream' \
                and ev['metadata'].get('langgraph_node') == 'assistant':
            toks.append(ev['data']['chunk'].content)
    return toks

toks = await tokens_do_assistant()
print('pedaços capturados:', len(toks), '| juntos:', repr(''.join(toks))[:80])
print('OK — o filtro isola o nó; com Groq real, os pedaços são os tokens.')

## 5. Erros clássicos

1. **Iterar o `stream` 2x esperando repetição.** Gerador esgota (CÉLULA 6).
   Converta em `list()` uma vez.
2. **Confundir `updates` com `values`.** Delta vs foto: para 'o que mudou agora'
   use updates; para 'como está tudo' use values.
3. **Esperar tokens no `updates`.** Updates carrega mensagens prontas por nó;
   token a token só em `messages` ou `astream_events`.
4. **Criar o Fake/LLM dentro do nó com loop.** Cada volta zera a sequência e o
   grafo nunca termina (Aulas 06 e 11 provaram). Construa fora, reuse dentro.
5. **Esquecer `thread_id` no `stream`.** Com checkpointer, `stream` exige
   `config` igual ao `invoke` — `ValueError` do mesmo jeito (Aula 06).
6. **`await` fora de contexto async.** Em `.py` use `asyncio.run()`; no Jupyter
   puro vale `await` direto na célula. Misturar dá erro de event loop.

## 6. Exercício guiado — contar os passos pelo `updates`

Situação: sem olhar o `invoke`, diga quantas passagens por nós a pergunta
'Some 3 e 4' provocou — e prove listando as chaves de cada chunk.

In [ ]:
# CÉLULA 9 — exercício guiado (conte os passos)
passos = [list(c.keys()) for c in reagente.stream(
    {'messages': [HumanMessage(content='Some 3 e 4')]}, 
    {'configurable': {'thread_id': 'u7'}},
    stream_mode='updates')]
print(passos)
assert passos == [['assistant'], ['tools'], ['assistant']], passos
print('OK — 3 passos: pedido, recibo, frase final.')

## 7. Desafio — 'digitando...' só do assistant

Monte um coletor que, via `stream_mode='messages'`, junte **só** os pedacinhos
do nó `assistant` e ignore os do `tools`:

- Com Groq real: imprima a frase final montada token a token e confira que o
  `'7'` do recibo **não** aparece no texto (ele veio do `tools`, filtrado fora).
- No Fake: confira que os chunks do `tools` existem no stream mas são pulados
  pelo filtro (`meta.get('langgraph_node') != 'assistant'`).
- **Pegadinha:** `ToolMessage` também passa pelo modo `messages` — filtrar por
  tipo (`AIMessageChunk`) sozinho não basta; a origem está no `metadata`.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que `invoke` devolve vs o que `stream` emite — e por que o gerador esvazia?
- [ ] Delta (`updates`), foto (`values`) e tupla+origem (`messages`): quando usar cada?
- [ ] Como isolar os tokens de UM nó no `astream_events`?
- [ ] Por que o LLM/Fake deve nascer fora do nó quando há loop?
- [ ] O que `thread_id` tem a ver com `stream` — e o que falta para pausar e retomar?

**Próxima aula (12 — Breakpoints):** pausar antes das tools para aprovação
humana (`interrupt_before`), inspecionar com `get_state().next` e retomar com
`stream(None, config)` — o primeiro humano-no-loop de verdade.